# 05 · Mixed Precision / Gradient Checkpointing —— 草稿纸对折用，步骤现推

**家族位置**：08 生产级优化第 5 站。01-04 省算/省搬/省参，本章省显存：AMP 半精度让激活/计算减半，梯度检查点让激活“现算现扔”，用时间换内存。

**学习目标**：AMP 的 fp32 主权+bfloat16 计算；检查点的前向不存/反向重算；内存×精度×耗时三口径；CPU 上无 GPU 加速的诚实声明。

## 1. 原理：对折草稿纸 + 现推步骤

### 通俗理解

**一句话**：AMP 像草稿纸对折——粗算（bfloat16）对折省纸，关键账（loss/权重）用原纸（fp32）保准头；检查点像考试只记小标题——需要哪步的草稿，现场重推一遍，桌子（显存）腾出来。

**比喻**：检查点 = `√K` 折中——K 层只存 K 层标题，反向走一层重推一层；比全存省内存，只多一次前向的时间。

### 结构账

```
AMP：autocast 粗算 bf16 + 主权重/loss 保持 fp32；CPU 上 bf16 可用、无加速（诚实口径）
检查点：use_reentrant=False；前向 saved_tensors 409MB→23MB（K=24 层块实测，×17.5）
口径：内存=子进程 Peak RSS（psutil，隔离 import 污染）+ saved_tensors 字节数
      精度=fp32/bf16 训练后 val-seq 差；耗时=CPU 秒表（不证 GPU 加速）
```

In [ ]:
import sys, time, copy
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from common.data import make_modadd_data
from common.engine import CkptMLP, activation_memory, measure_peak, task_ckpt_train
from common.models import ToyGPT
from common.utils import set_seed,setup_chinese_font,count_params
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
print('torch:',torch.__version__)
print('CUDA:',torch.cuda.is_available(),'| CPU bf16 autocast: 可用（无加速，保兼容口径）')
Xt,yt=make_modadd_data(4000,16,16,seed=10); Xw,yw=make_modadd_data(500,16,16,seed=11)
tr=DataLoader(TensorDataset(Xt,yt),batch_size=128,shuffle=True); va=DataLoader(TensorDataset(Xw,yw),batch_size=512)
print(f'modadd train {len(tr.dataset)} / val {len(va.dataset)} | S=16 vocab=16')

## 2. 检查点：激活内存 409MB→23MB（×17.5）

In [ ]:
torch.manual_seed(0)
mlp=CkptMLP(dim=512,depth=24)
src=torch.randn(8,16,512)
mem_off,loss1=activation_memory(mlp,src,use_ckpt=False)
mem_on,loss2=activation_memory(mlp,src,use_ckpt=True)
print(f'saved tensors: full={mem_off/1e6:.1f}MB ckpt={mem_on/1e6:.1f}MB 压缩×{mem_off/mem_on:.1f}')
print(f'前向等价: |Δloss|={abs(loss1.item()-loss2.item()):.2e}')
fig,ax=plt.subplots(figsize=(6,3.2))
ax.bar(['full','checkpoint'],[mem_off/1e6,mem_on/1e6],color=['#DD8452','#4C72B0'])
for i,v in enumerate([mem_off/1e6,mem_on/1e6]): ax.text(i,v+8,f'{v:.0f}MB',ha='center',fontsize=10)
ax.set_ylabel('saved tensors (MB)'); ax.set_title('梯度检查点：K=24 层激活压缩')
plt.tight_layout(); plt.savefig(FIGS/'fig1_ckpt_mem.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. 端到端峰值内存：子进程 Peak RSS 隔离测量

In [ ]:
_,peak_off=measure_peak(task_ckpt_train,False)
_,peak_on=measure_peak(task_ckpt_train,True)
print(f'peak RSS: full={peak_off/1e6:.0f}MB ckpt={peak_on/1e6:.0f}MB 省×{peak_off/peak_on:.2f}')
fig,ax=plt.subplots(figsize=(6,3.0))
ax.bar(['full','checkpoint'],[peak_off/1e6,peak_on/1e6],color=['#DD8452','#4C72B0'])
for i,v in enumerate([peak_off/1e6,peak_on/1e6]): ax.text(i,v+10,f'{v:.0f}MB',ha='center',fontsize=10)
ax.set_ylabel('peak RSS (MB)'); ax.set_title('端到端训练峰值内存（子进程隔离）')
plt.tight_layout(); plt.savefig(FIGS/'fig2_peak.png',dpi=150,bbox_inches='tight'); plt.show()

## 4. AMP bf16：精度口径 + 训练对照

In [ ]:
def seq_acc(m):
    m.eval()
    with torch.no_grad():
        ok=tot=0
        for src,tgt in va:
            pred=m(src).argmax(-1)
            ok+=(pred==tgt).all(1).sum().item(); tot+=len(src)
    return ok/tot
def train_gpt(dtype):
    torch.manual_seed(0)
    m=ToyGPT(vocab=16,dim=64,depth=2,heads=4,mode='sincos')
    opt=torch.optim.Adam(m.parameters(),lr=3e-3)
    hist=[]
    for ep in range(30):
        m.train(); tot=0
        for src,tgt in tr:
            if dtype==torch.float32:
                logits=m(src); loss=nn.functional.cross_entropy(logits.reshape(-1,16),tgt.reshape(-1))
            else:
                with torch.autocast('cpu',dtype=dtype):
                    logits=m(src)
                loss=nn.functional.cross_entropy(logits.float().reshape(-1,16),tgt.reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()*len(src)
        hist.append(tot/len(tr.dataset))
    return m,hist
res={}
for name,dt in [('fp32',torch.float32),('bf16',torch.bfloat16)]:
    t0=time.perf_counter(); m,h=train_gpt(dt); el=time.perf_counter()-t0
    a=seq_acc(m); res[name]=(a,el,h)
    print(f'{name}: val-seq={a:.4f} loss={h[-1]:.4f} time={el:.1f}s',flush=True)
fig,ax=plt.subplots(1,2,figsize=(9,3.2))
for name,c in [('fp32','#4C72B0'),('bf16','#DD8452')]:
    ax[0].plot(res[name][2],label=name,color=c)
ax[0].set_title('CE loss'); ax[0].legend()
ax[1].bar(res.keys(),[res[k][0] for k in res],color=['#4C72B0','#DD8452'])
for i,k in enumerate(res): ax[1].text(i,res[k][0]+0.01,f'{res[k][0]:.3f}',ha='center')
ax[1].set_ylim(0,1.05); ax[1].set_title('val-seq：bf16 精度不掉')
plt.tight_layout(); plt.savefig(FIGS/'fig3_amp.png',dpi=150,bbox_inches='tight'); plt.show()

## 5. 耗时口径：检查点用时间换内存的实测

In [ ]:
src=torch.randn(8,16,512); tgt=torch.randint(0,16,(8,16))
opt=torch.optim.Adam(mlp.parameters(),lr=1e-3)
for ck in [False,True]:
    ts=[]
    for _ in range(3):
        t0=time.perf_counter()
        logits=mlp(src,use_ckpt=ck)
        loss=nn.functional.cross_entropy(logits.reshape(-1,16),tgt.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        ts.append(time.perf_counter()-t0)
    print(f'use_ckpt={ck}: {min(ts)*1000:.0f}ms/step（多一次前向的代价）')
print('总结：检查点激活×17.5 + 峰值RSS实测；AMP bf16 精度不掉；CPU 无 GPU 加速（诚实口径）。下一步 06 量化/剪枝。')